# Swiss Dataset - ejercicio resuelto y explicado

Este notebook acompaña la resolución del ejercicio `swiss_reg.ipynb`. La idea es entender qué hace cada paso, por qué se usa cada herramienta y cómo interpretar los resultados.

## 1. Problema de aprendizaje supervisado

Queremos predecir `Fertility`, una variable numérica. Por eso el problema es de **regresión**. Usamos como atributos las columnas numéricas restantes: `Agriculture`, `Examination`, `Education`, `Catholic` e `Infant.Mortality`.

`Location` no se usa porque es una etiqueta de texto de la provincia, no una variable numérica directa para estos modelos.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("swiss.csv")
df.head()

In [ ]:
X = df.drop(["Fertility", "Location"], axis=1)
y = df["Fertility"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42
)

print(f"Entrenamiento: {X_train.shape[0]} observaciones")
print(f"Test: {X_test.shape[0]} observaciones")

## 2. Por qué escalamos

`StandardScaler` transforma cada atributo para que tenga media 0 y desvío estándar 1. Esto es especialmente importante en modelos regularizados, porque la penalización compara coeficientes: si una variable está en una escala mucho mayor que otra, la regularización puede castigar de forma desbalanceada.

Usamos `Pipeline` para que el escalado y el modelo queden unidos. Así evitamos errores y nos aseguramos de que el `StandardScaler` se ajuste solo con los datos de entrenamiento.

## 3. Regresión lineal múltiple

La regresión lineal múltiple aprende una combinación lineal de todos los atributos disponibles. No elimina variables por sí misma: intenta usar todas para minimizar el error cuadrático en entrenamiento.

In [ ]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, y_pred_linear)
linear_mse = mean_squared_error(y_test, y_pred_linear)
linear_r2 = r2_score(y_test, y_pred_linear)

print(f"MAE: {linear_mae:.4f}")
print(f"MSE: {linear_mse:.4f}")
print(f"R2: {linear_r2:.4f}")

## 4. Métricas

**MAE** mide el error absoluto promedio. Es fácil de interpretar porque está en la misma unidad que `Fertility`.

**MSE** promedia errores al cuadrado. Penaliza más los errores grandes.

**R²** mide qué proporción de la variabilidad de la variable objetivo explica el modelo. Un valor más alto suele indicar mejor ajuste, aunque siempre hay que mirarlo junto con las otras métricas.

## 5. Regularización y selección de variables

La consigna pide elegir el método capaz de hacer selección automática de variables. Ese método es **Lasso**, porque usa regularización L1.

Lasso puede llevar algunos coeficientes exactamente a cero. Cuando eso pasa, el modelo deja de usar esas variables. Ridge, en cambio, usa L2: reduce coeficientes, pero normalmente no los anula por completo.

In [ ]:
alphas = np.logspace(-4, 1, 500)

lasso_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LassoCV(alphas=alphas, cv=3, random_state=42, max_iter=10000))
])

lasso_model.fit(X_train, y_train)
y_pred_lasso = lasso_model.predict(X_test)

lasso_mae = mean_absolute_error(y_test, y_pred_lasso)
lasso_mse = mean_squared_error(y_test, y_pred_lasso)
lasso_r2 = r2_score(y_test, y_pred_lasso)

best_alpha = lasso_model.named_steps["regressor"].alpha_
coefs = pd.Series(
    lasso_model.named_steps["regressor"].coef_,
    index=X.columns,
    name="coeficiente"
)

print(f"Mejor alpha: {best_alpha:.4f}")
print(f"MAE: {lasso_mae:.4f}")
print(f"MSE: {lasso_mse:.4f}")
print(f"R2: {lasso_r2:.4f}")
coefs.to_frame()

## 6. Comparación final

Con este split de entrenamiento y test, Lasso mejora apenas las tres métricas. La diferencia no es enorme, pero además entrega un modelo más simple porque elimina `Agriculture` al asignarle coeficiente cero.

In [ ]:
resultados = pd.DataFrame({
    "Modelo": ["Regresión Lineal", "Lasso (L1)"],
    "MAE": [linear_mae, lasso_mae],
    "MSE": [linear_mse, lasso_mse],
    "R2": [linear_r2, lasso_r2],
})

resultados

## 7. Lectura conceptual

El ejercicio muestra un flujo típico de Machine Learning supervisado: separar datos, entrenar modelos, predecir sobre test y comparar métricas.

La regresión lineal sirve como modelo base. Lasso agrega regularización L1, busca automáticamente el mejor `alpha` con validación cruzada de 3 folds y puede simplificar el modelo eliminando atributos poco útiles.

En datasets pequeños como este, las métricas pueden cambiar bastante si cambia el split. Por eso es importante fijar `random_state` para reproducibilidad y usar validación cruzada cuando estamos ajustando hiperparámetros.